In [1]:
from pathlib import Path
import json
import pandas as pd

from benchmark_runner import (
    run_benchmark
)

In [2]:
benchmark_configs = {

    "FIN379": {

        "data_file":
            Path(
                "../validation/benchmark_systems/FIN379/FIN379.inp"
            ),

        "expected_file":
            Path(
                "../validation/expected_results/FIN379.json"
            )
    },

    "GL765": {

        "data_file":
            Path(
                "../validation/benchmark_systems/GL765/GL765.inp"
            ),

        "expected_file":
            Path(
                "../validation/expected_results/GL765.json"
            )
    },

    "ZetaAqr": {

        "data_file":
            Path(
                "../validation/benchmark_systems/ZetaAqr/ZetaAqr.in2"
            ),

        "expected_file":
            Path(
                "../validation/expected_results/ZetaAqr.json"
            )
    }
}

In [3]:
results = {}

for system, config in benchmark_configs.items():

    print(f"Running {system}")

    results[system] = run_benchmark(
        system,
        config
    )

print()

print("Done.")

Running FIN379
Running GL765
Running ZetaAqr

Done.


In [4]:
rows = []

for system, result in results.items():

    stats = result["statistics"]

    rows.append({

        "System": system,

        "Reduced Chi2":
            stats["reduced_chi2"],

        "RMS":
            stats.get("rms", float("nan")),

        "Success":
            result["optimizer_success"]
    })

summary = pd.DataFrame(rows)

summary

,System,Reduced Chi2,RMS,Success
0,FIN379,2.625809,1.476739,True
1,GL765,3.819334,1.863363,True
2,ZetaAqr,5.874282,2.410606,True


In [5]:
rows = []

for system, result in results.items():

    for parameter, values in (
        result["parameter_errors"].items()
    ):

        rows.append({

            "System": system,

            "Parameter": parameter,

            "Relative Error":
                values["relative_error"]
        })

errors = pd.DataFrame(rows)

errors

,System,Parameter,Relative Error
0,FIN379,P,0.002760
1,FIN379,T0,0.000004
2,FIN379,e,0.007923
3,FIN379,a,0.008997
4,FIN379,Omega,0.978333
5,FIN379,omega,0.980604
6,FIN379,i,0.012330
7,FIN379,K1,0.004212
8,FIN379,K2,0.003961
9,FIN379,V0,0.006210


In [6]:
ANGLE_PARAMETERS = [
    "Omega",
    "omega"
]

non_angle_errors = errors[
    ~errors["Parameter"].isin(
        ANGLE_PARAMETERS
    )
]

non_angle_errors

,System,Parameter,Relative Error
0,FIN379,P,0.002760
1,FIN379,T0,0.000004
2,FIN379,e,0.007923
3,FIN379,a,0.008997
6,FIN379,i,0.012330
7,FIN379,K1,0.004212
8,FIN379,K2,0.003961
9,FIN379,V0,0.006210
10,GL765,P,0.006195
11,GL765,T0,0.000081


In [7]:
rows = []

for system in (
    non_angle_errors["System"].unique()
):

    system_errors = non_angle_errors[
        non_angle_errors["System"] == system
    ]

    rows.append({

        "System": system,

        "Mean Relative Error":
            system_errors[
                "Relative Error"
            ].mean(),

        "Max Relative Error":
            system_errors[
                "Relative Error"
            ].max()
    })

benchmark_summary = pd.DataFrame(rows)

benchmark_summary

,System,Mean Relative Error,Max Relative Error
0,FIN379,0.005800,0.012330
1,GL765,0.111667,0.659914
2,ZetaAqr,0.050926,0.103707


In [8]:
from pathlib import Path

Path(
    "../validation/results"
).mkdir(
    parents=True,
    exist_ok=True
)

In [9]:
benchmark_summary.to_csv(
    "../validation/results/benchmark_summary.csv",
    index=False
)

print(
    "Benchmark summary saved."
)

Benchmark summary saved.


In [10]:
from pathlib import Path
import numpy as np

from pyseshat.io.orbit_parser import (
    parse_orbit_file
)

from pyseshat.orbit.orbit_model import (
    sample_visual_orbit
)

from pyseshat.inference.optimizer import (
    fit_visual_orbit
)

from pyseshat.visualization.orbit_plot import (
    plot_visual_orbit
)

In [11]:
FIGURE_DIR = Path(
    "../validation/results/figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DIR

WindowsPath('../validation/results/figures')

In [12]:
for system, config in benchmark_configs.items():

    print(f"Generating {system}")

    parsed = parse_orbit_file(
        str(config["data_file"])
    )

    elements = parsed["elements"]

    visual_data = parsed["visual_data"]

    rv_data = parsed["rv_data"]

    # ----------------------------------
    # Convert observations
    # ----------------------------------

    t = visual_data[:,0]

    theta = np.deg2rad(
        visual_data[:,1]
    )

    rho = visual_data[:,2]

    sigma = visual_data[:,3]

    X = rho * np.cos(theta)

    Y = rho * np.sin(theta)

    # ----------------------------------
    # Fit
    # ----------------------------------

    fit_result = fit_visual_orbit(
        elements,
        t,
        X,
        Y,
        sigma,
        sigma,
        rv_data=rv_data
    )

    fitted = fit_result.elements

    # ----------------------------------
    # Smooth orbit
    # ----------------------------------

    orbit = sample_visual_orbit(
        fitted,
        n_points=1000
    )

    # ----------------------------------
    # Plot
    # ----------------------------------

    fig, ax = plot_visual_orbit(
        X,
        Y,
        orbit,
        title=system,
        show=False
    )

    outfile = (
        FIGURE_DIR
        /
        f"{system}_orbit.png"
    )

    fig.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight"
    )

    print(outfile)

    fig.clf()

Generating FIN379
..\validation\results\figures\FIN379_orbit.png
Generating GL765
..\validation\results\figures\GL765_orbit.png
Generating ZetaAqr
..\validation\results\figures\ZetaAqr_orbit.png


<Figure size 700x700 with 0 Axes>

<Figure size 700x700 with 0 Axes>

<Figure size 700x700 with 0 Axes>

GL765 is currently identified as a challenging combined visual+RV benchmark.

The visual-only fit reproduces the observed orbit geometry well.
The combined fit converges successfully but exhibits a degraded
astrometric solution relative to the reference orbit.

This benchmark is retained to track future improvements in
multimodal optimization and weighting strategies.

In [13]:
gl = results["GL765"]

for p, v in gl["parameter_errors"].items():

    print(
        f"{p:8s}",
        f"{v['relative_error']:.4f}"
    )

P        0.0062
T0       0.0001
e        0.0380
a        0.6599
Omega    0.5989
omega    0.9906
i        0.1487
K1       0.0145
K2       0.0231
V0       0.0029


In [14]:
from pathlib import Path
import pandas as pd

rows = []

for system, result in results.items():

    stats = result["statistics"]

    parameter_errors = [

        v["relative_error"]

        for k, v in (
            result["parameter_errors"].items()
        )

        if k not in [
            "Omega",
            "omega"
        ]
    ]

    rows.append({

        "system": system,

        "success": result[
            "optimizer_success"
        ],

        "reduced_chi2": stats[
            "reduced_chi2"
        ],

        "rms": stats[
            "rms"
        ],

        "mean_relative_error":
            sum(parameter_errors)
            / len(parameter_errors),

        "max_relative_error":
            max(parameter_errors)
    })

benchmark_report = pd.DataFrame(rows)

benchmark_report

,system,success,reduced_chi2,rms,mean_relative_error,max_relative_error
0,FIN379,True,2.625809,1.476739,0.005800,0.012330
1,GL765,True,3.819334,1.863363,0.111667,0.659914
2,ZetaAqr,True,5.874282,2.410606,0.050926,0.103707


In [15]:
import json

OUTPUT_DIR = Path(
    "../validation/results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ----------------------------------
# CSV summary
# ----------------------------------

benchmark_report.to_csv(

    OUTPUT_DIR
    /
    "benchmark_summary.csv",

    index=False
)

# ----------------------------------
# JSON report
# ----------------------------------

json_report = {}

for system, result in results.items():

    json_report[system] = {

        "optimizer_success":
            result["optimizer_success"],

        "optimizer_message":
            result["optimizer_message"],

        "statistics":
            result["statistics"],

        "parameter_errors":
            result["parameter_errors"]
    }

with open(

    OUTPUT_DIR
    /
    "benchmark_report.json",

    "w"

) as f:

    json.dump(
        json_report,
        f,
        indent=2
    )

print(
    "Saved benchmark_summary.csv"
)

print(
    "Saved benchmark_report.json"
)

Saved benchmark_summary.csv
Saved benchmark_report.json
